# Test 1, Part 2 — Applied Regression

## Introduction

This is the take-home portion of Test 1. Part 1 assessed your conceptual understanding of regression, cross-validation, and regularization. Part 2 assesses whether you can *apply* those ideas independently.

You are given a dataset with pre-completed exploratory analysis. Your job is to build, evaluate, and compare regression models, then interpret your findings. The emphasis is on interpretation — working code that produces results without explanation will receive minimal credit. Clear reasoning with minor code issues will receive substantial credit.

### Rules

- Open book, open notes, open documentation
- GenAI tools allowed with disclosure (same policy as homework)
- Individual work — do not collaborate with classmates
- Due: Wednesday, March 18 (flexible by request)

### Scoring (26 points + 3 bonus)

| Component | Points |
|-----------|-------:|
| Baseline | 2 |
| Pipeline + CV models | 4 |
| Model comparison | 3 |
| Hyperparameter tuning | 4 |
| Regularization analysis | 6 |
| Model recommendation | 4 |
| New method reflection | 3 |
| **Total** | **26** |
| *Bonus: visualizations* | *+3* |

Interpretation prompts (regularization analysis, model recommendation, new method reflection) account for 50% of the grade. Bonus points are available for visualizations that clearly support your analysis — e.g., coefficient comparison plots, hyperparameter sweep curves, or model comparison charts.

### Time Estimate

This should take under 3 hours. If you find yourself spending significantly more time, step back and focus on interpretation rather than perfecting code.

### Tips

- Don't worry about fancy formatting. Grading is based on content, not form.
- The required modeling portion of this assignment (data prep, baseline, pipelines, CV, tuning, and coefficient extraction) is easily achieved in fewer than 200 lines of Python. This is not a contest for fewest (or most) lines - it is just a calibration point. If you are well past that and over budget on time, consider whether you are overcomplicating things.
- You may use manual loops with `cross_val_score` or `GridSearchCV` for hyperparameter tuning - either approach is fine.

## The Data

The National Extraterrestrial Event Registry (NEER) has compiled data on 200 confirmed alien abduction events across the continental United States. For each event, field agents recorded environmental conditions, geographic measurements, and local demographic indicators. Your task is to predict the **duration of the abduction** (in minutes) from these features.

### Data Dictionary

| Feature | Description | Units |
|---------|-------------|-------|
| `em_base` | Baseline electromagnetic reading | mV |
| `em_anom` | Electromagnetic anomaly index | mV |
| `em_flux` | Electromagnetic flux intensity | mV |
| `em_freq` | Electromagnetic frequency deviation | Hz |
| `em_peak` | Peak electromagnetic signal | mV |
| `geo_lat` | Latitude of event | degrees N |
| `geo_rural` | Rurality index | composite score |
| `geo_elev` | Elevation | ft |
| `corn_prox` | Proximity to nearest cornfield | km |
| `cow_pop` | Local cow population | head count |
| `lunar_lum` | Lunar luminosity at time of event | 0-1 scale |
| `atmo_pres` | Atmospheric pressure | hPa |
| `temp_f` | Temperature | °F |
| `tin_foil` | Local tin foil hat density | hats/km² |
| `radio_vol` | Ambient radio volume | dB |
| `shoe_size` | Regional average shoe size | US size |
| `dog_bark` | Dog barks per hour in vicinity | count/hr |
| `wifi_str` | WiFi signal strength | dBm |
| `bird_cnt` | Bird count in area | count |
| `wind_dir` | Wind direction | degrees |
| `humidity` | Relative humidity | % |
| `uv_index` | UV index | 0-11 scale |
| `traffic` | Local traffic density | vehicles/hr |
| **`duration`** | **Abduction duration (target)** | **minutes** |

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
url = "https://raw.githubusercontent.com/olearydj/INSY7120/refs/heads/main/tests/test1-part2-data.csv"
df = pd.read_csv(url)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.describe().round(2)

## Exploratory Analysis

The following EDA is provided for you. Read it, but do not spend time on additional exploration — focus your effort on modeling and interpretation.

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["duration"], bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("Abduction Duration (minutes)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Abduction Duration")
ax.axvline(df["duration"].mean(), color="red", linestyle="--", label=f"Mean: {df['duration'].mean():.1f}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            annot_kws={"size": 7})
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Correlations with target, sorted
target_corr = df.corr()["duration"].drop("duration").sort_values(key=abs, ascending=False)
print("Correlation with duration:")
print(target_corr.round(3).to_string())

In [ ]:
# Scatter plots for top predictors
top_features = target_corr.abs().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, top_features):
    ax.scatter(df[feat], df["duration"], alpha=0.4, s=15)
    ax.set_xlabel(feat)
    ax.set_ylabel("duration")
    ax.set_title(f"r = {df[feat].corr(df['duration']):.3f}")
plt.suptitle("Top 6 Features by Correlation with Duration")
plt.tight_layout()
plt.show()

### NEER Field Brief and Observations

Senior agents have observed that abductions tend to last longer in rural areas, particularly near agricultural land. Electromagnetic disturbances seem to be the most reliable predictor of extraterrestrial activity, though our instruments record several overlapping EM signals that may capture the same underlying phenomenon. Geographic factors are mathematically correlated with each other and their individual contributions are debated internally.

The EDA above should confirm several of these observations. Review the correlation heatmap and scatter plots carefully — the patterns you see here should inform your modeling choices and will be important for interpretation later.

Note that the dataset has 23 features and only 200 observations. Consider what this ratio implies for model selection.

## Your Work Starts Here

Prepare the data for modeling, then build and evaluate regression models. Your submission must demonstrate the following:

1. **Data preparation** — Separate features and target. Create a train-test split.

2. **Baseline** — Establish a performance floor using `DummyRegressor`.

3. **Model comparison** — Evaluate at least 3 of the following models using `Pipeline` and cross-validation: `LinearRegression` (OLS), `Ridge`, `Lasso`, `ElasticNet`. All preprocessing must be inside the Pipeline. Report CV scores (mean and standard deviation). Present a clear comparison of all models.

4. **Hyperparameter tuning** — For your recommended model, tune at least 2 hyperparameters. Examples: regularization strength (`alpha`), polynomial degree (`PolynomialFeatures`), interaction terms, `l1_ratio` for Elastic Net. Show how performance varies with each hyperparameter and justify your final choices.

5. **Coefficient analysis** — Extract and compare the coefficients from your OLS and regularized models. You will need this for the interpretation prompts.

6. **Interpretation** — Answer the interpretation prompts below. This is where most of the points are.

## Interpretation Prompts

Answer each of the following after completing your modeling work. Concise, evidence-based responses are better than lengthy ones.

### Regularization Analysis (6 pts)

Compare the coefficients of your OLS and regularized models. What does the comparison reveal about the predictor structure of this dataset? Which features does regularization treat differently, and why? Pay particular attention to the electromagnetic and geographic feature groups.

---

*Your response here.*

---

### Model Recommendation (4 pts)

Which model do you recommend for this dataset? What evidence supports your choice? Consider CV scores, the train-test gap, parsimony, and what the model tells you about the data. What would you try next if you had more time?

---

*Your response here.*

---

### New Method Reflection (3 pts)

If you used Elastic Net: how does it relate to Ridge and Lasso? What does the `l1_ratio` parameter control, and what did your tuning reveal about the best balance for this dataset?

If you used `PolynomialFeatures` inside a Pipeline: what happened when you added polynomial terms without regularization? With regularization? What does this tell you about the interaction between feature expansion and regularization?

If you used another approach: what method did you choose, how did you learn to use it, and how does it connect to what we covered in class?

---

*Your response here.*

---

## Reflection

1. **GenAI use** — If you used AI tools, specify which tool/model, how you used it, and how you verified correctness. If not, briefly note why.

2. **Time spent** — Approximately how long did this take?

---

*Your response here.*

---